### Step 1: Import Libraries and Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

import requests
import json
import random

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None: 
    raise Exception("OpenAI API Key is not set!")

client = OpenAI(api_key=OPENAI_API_KEY)

load_dotenv()

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

### Step 2: Create tools and function calling

In [2]:
client = OpenAI()

# Function that sends notification via pushover app
def send_notification(message: str):
    payload = {'user': pushover_user, 'token': pushover_token, 'device': 'oneplusnordn2005g', 'message': message}
    requests.post(pushover_url, data=payload)
    return f"Notification message was sent: {message}"

# Function that converts message to emoji symbols (NOT WORKING)
def convert_message_to_emoji(message: str):
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.responses.create(
        model = 'gpt-4o-mini',
        input = f"Convert this message into EXCLUSIVELY EMOJIS: {message}"
    )
    return response.output_text

# Function simulating roling a single six-sided die
def dice_roll():
    return random.randint(1,6)

# DESCRIBE THE FUNCTIONS TO THE LLM

dice_roll_function = {
    'name': 'dice_roll',
    'description': 'Simulates rolling a single six-sided die and returns the result of that roll. Use this when the user wants to roll a die for games, decisions, or random numbers.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': []
    }
}

send_notification_function = {
    'name': 'send_notification',
    'description': """
                Sends a message to send as a push notification to the users phone via Pushover. 
                User this to alert the user about a new message.""",
    'parameters': {
        'type': 'object',
        'properties': {
            'message': {
                'type': 'string',
                'description': 'The notification message to send to the users device'
            }
        },
        "required": ["message"]
    }
}

convert_message_to_emoji_function = {
    'name': 'convert_message_to_emoji',
    'description': "When the last message has emotional language, particularly at least one emoji, convert the ENTIRE MESSAGE TEXT into a series of vibrant emoji symbols.",
    'parameters': {
        'type': 'object',
        'properties': {
            'message': {
                'type': 'string',
                'description': 'The message which has emotional language and at least one emoji'
            }
        },
        "required": ["message"]
    }
}


tools = [
    {"type":"function", "function": send_notification_function},
    {"type":"function", "function": convert_message_to_emoji_function},
    {"type":"function", "function": dice_roll_function}
]


In [3]:
def handle_tool_call(tool_calls):
    tool_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f"Calling fucnction {function_name}") #For future debuggins

        if function_name == 'send_notification':
            content = send_notification(args['message'])
        elif function_name == 'inspire_emoji':
            content = convert_message_to_emoji(args['message'])
        elif function_name == 'dice_roll':
            content = f"Dice roll was: {dice_roll()}"
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        tool_results.append(tool_call_result)
    return tool_results


In [7]:
from pprint import pprint 

client = OpenAI()
messages = [
    {"role": "user", "content": """
            Please do two things: \
             1) I want to roll 4 dice separately; tell me each roll, and \
            2) Send me a notification with with the highest of the rolls!
     """}
                
]

response = client.chat.completions.create(
    model = 'gpt-4.1-mini',
    messages = messages,
    temperature=0.9,
    tools=tools
)

#Check if model wants to call a tool
resp = response.choices[0].message
# print(message)

while resp.tool_calls:
    pprint(resp.tool_calls)
    # handle tool call
    tool_result = handle_tool_call(resp.tool_calls)

    messages.append(resp)
    messages.extend(tool_result)

    # invoke the LLM one more time to get updated response
    response = client.chat.completions.create(
        model = 'gpt-4.1-mini',
        messages = messages,
        tools = tools
    )
    resp = response.choices[0].message


print(resp.content)

[ChatCompletionMessageFunctionToolCall(id='call_zb5Gh37hQdVmNEih14wQSeVU', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_7yMuP3z3W3PqSfOUGYu20SLD', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_LnbvbH5NaJmQFMf0ohPxCPE8', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_b4yukFAEx6PEpqDkiKYQyyzY', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_A9s6XfiLzzTrI7UfqMTJCXc5', function=Function(arguments='{"message":"The highest roll among your four dice rolls is 6."}', name='send_notification'), type='function')]
Here are your four dice rolls: 6, 3, 5, and 4. I have also sent you a notification with the highest roll, which is 6.
